In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [2]:
data = pd.read_csv('TSA_Dataset.csv')
data = data.dropna(subset=['text', 'sentiment']).copy()
data['sentiment'] = data['sentiment'].astype(int)
data.head()


,text,sentiment
0,i just feel really helpless and heavy hearted,0
1,ive enjoyed being able to slouch about relax a...,0
2,i gave up my internship with the dmrg and am f...,0
3,i dont know i feel so lost,0
4,i am a kindergarten teacher and i am thoroughl...,0


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ''

    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


data['cleaned_text'] = data['text'].apply(clean_text)
data = data[data['cleaned_text'].str.len() > 0].copy()
data.head()


,text,sentiment,cleaned_text
0,i just feel really helpless and heavy hearted,0,i just feel really helpless and heavy hearted
1,ive enjoyed being able to slouch about relax a...,0,ive enjoyed being able to slouch about relax a...
2,i gave up my internship with the dmrg and am f...,0,i gave up my internship with the dmrg and am f...
3,i dont know i feel so lost,0,i dont know i feel so lost
4,i am a kindergarten teacher and i am thoroughl...,0,i am a kindergarten teacher and i am thoroughl...


In [4]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Keep words that carry sentiment. Removing these can flip the meaning of a sentence.
for word in ['not', 'no', 'nor', 'never', 'very', 'too']:
    stop_words.discard(word)


def lemmatize_text(text):
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)


data['cleaned_text'] = data['cleaned_text'].apply(lemmatize_text)
data = data[data['cleaned_text'].str.len() > 0].copy()
data[['text', 'sentiment', 'cleaned_text']].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pawan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\pawan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\pawan\AppData\Roaming\nltk_data...


,text,sentiment,cleaned_text
0,i just feel really helpless and heavy hearted,0,feel really helpless heavy hearted
1,ive enjoyed being able to slouch about relax a...,0,ive enjoyed able slouch relax unwind frankly n...
2,i gave up my internship with the dmrg and am f...,0,gave internship dmrg feeling distraught
3,i dont know i feel so lost,0,dont know feel lost
4,i am a kindergarten teacher and i am thoroughl...,0,kindergarten teacher thoroughly weary job take...


In [5]:
data.to_csv('Clean_Data.csv', index=False)
print(f"Saved {len(data)} cleaned rows to Clean_Data.csv")
print(data['sentiment'].value_counts().sort_index())


Saved 17755 cleaned rows to Clean_Data.csv
sentiment
0    5723
1    6237
2    5795
Name: count, dtype: int64
